# Sesión 4: Consultas Agrupadas (Parte I)
## GROUP BY y Funciones de Agregación: MIN, MAX, AVG, COUNT

**Módulo:** Fundamentos de Programación Python para el Análisis de Datos

**Contenido:**
- Funciones de agregación: MIN(), MAX(), AVG(), COUNT()
- Cláusula GROUP BY para agrupamiento de datos
- Interpretación y validación de resultados agregados
- Errores comunes y buenas prácticas

## Configuración Inicial

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

# Crear conexión
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print("✓ Conexión SQLite establecida")

## SLIDE 3: Desafío Inicial

**Contexto:** Analista en entidad gubernamental que monitorea programas de capacitación.

**Solicitud:** Generar reporte que muestre:
1. Promedio de calificaciones por programa
2. Cantidad de estudiantes por modalidad
3. Calificación mínima y máxima por programa
4. Programas con más de 2 estudiantes

**Pregunta:** ¿Cómo resumir grandes volúmenes de datos en indicadores útiles?

In [ ]:
# SLIDE 13-14: Crear tabla base
cursor.execute('''
CREATE TABLE evaluaciones (
    estudiante VARCHAR(50),
    programa VARCHAR(50),
    modalidad VARCHAR(20),
    calificacion DECIMAL(3, 1)
)
''')

# Insertar datos
datos = [
    ('Ana García', 'Python Avanzado', 'Presencial', 8.5),
    ('Bruno López', 'Python Avanzado', 'Presencial', 9.0),
    ('Carlos Martín', 'Python Avanzado', 'Virtual', 7.5),
    ('Diana Ruiz', 'SQL Empresarial', 'Presencial', 8.0),
    ('Elena Sánchez', 'SQL Empresarial', 'Virtual', 8.5),
    ('Fiona Chen', 'SQL Empresarial', 'Virtual', 7.0),
    ('Gabriel López', 'Python Avanzado', 'Virtual', 8.0),
    ('Héctor Ruiz', 'Cloud Computing', 'Presencial', 9.5),
    ('Isabel Martín', 'Cloud Computing', 'Presencial', 8.5),
    ('Javier García', 'Cloud Computing', 'Virtual', 7.5),
    ('Karen López', 'Big Data', 'Virtual', 9.0),
    ('Luis Pérez', 'Big Data', 'Virtual', 8.5)
]

cursor.executemany(
    'INSERT INTO evaluaciones VALUES (?, ?, ?, ?)',
    datos
)

conn.commit()
print("✓ Tabla 'evaluaciones' creada con 12 registros\n")

# Ver datos
df = pd.read_sql('SELECT * FROM evaluaciones', conn)
print("Contenido de la tabla:")
print(df)

## SLIDE 4: Funciones de Agregación en SQL

**¿Qué son?** Funciones que operan sobre un conjunto de datos y devuelven un solo valor.

**Funciones principales:**
- MIN(): Valor mínimo
- MAX(): Valor máximo
- AVG(): Promedio aritmético
- COUNT(): Cantidad de registros

## SLIDE 5-6: MIN() y MAX()

**MIN()** devuelve el valor mínimo. **MAX()** devuelve el máximo.

**Contexto productivo:** Útiles para monitorear extremos de rendimiento, fechas límite, rangos de precios.

In [ ]:
# Ejemplos básicos de MIN y MAX
print("="*70)
print("SLIDE 5-6: MIN() y MAX()")
print("="*70 + "\n")

# Calificación mínima y máxima global
query = '''
SELECT 
    MIN(calificacion) as calif_minima,
    MAX(calificacion) as calif_maxima,
    MAX(calificacion) - MIN(calificacion) as rango
FROM evaluaciones
'''

print("Consulta: Calificación mínima, máxima y rango global")
print(query)
print("\nResultado:")
df_minmax = pd.read_sql(query, conn)
print(df_minmax.to_string())

In [ ]:
# MIN/MAX por programa (SIN GROUP BY aún - esto es incorrecto en algunos motores)
print("\n" + "-"*70)
print("MIN/MAX por programa (Veremos GROUP BY después)")
print("-"*70 + "\n")

query_programa = '''
SELECT 
    programa,
    MIN(calificacion) as calif_minima,
    MAX(calificacion) as calif_maxima
FROM evaluaciones
GROUP BY programa
ORDER BY programa
'''

print("Consulta con GROUP BY:")
print(query_programa)
print("\nResultado:")
df_minmax_prog = pd.read_sql(query_programa, conn)
print(df_minmax_prog.to_string())

## SLIDE 7-8: AVG()

**AVG()** calcula el promedio aritmético de una columna numérica.

**Contexto productivo:** Muy usada en análisis de desempeño académico, satisfacción de clientes, productividad.

In [ ]:
print("\n" + "="*70)
print("SLIDE 7-8: AVG()")
print("="*70 + "\n")

# Promedio global
query_avg = '''
SELECT 
    ROUND(AVG(calificacion), 2) as promedio_global,
    COUNT(*) as total_evaluaciones
FROM evaluaciones
'''

print("Consulta: Promedio global de calificaciones")
print(query_avg)
print("\nResultado:")
df_avg = pd.read_sql(query_avg, conn)
print(df_avg.to_string())

In [ ]:
# Promedio por programa
query_avg_prog = '''
SELECT 
    programa,
    ROUND(AVG(calificacion), 2) as promedio,
    COUNT(*) as total_estudiantes
FROM evaluaciones
GROUP BY programa
ORDER BY promedio DESC
'''

print("\n" + "-"*70)
print("Promedio de calificaciones por programa:")
print("-"*70)
print("\nConsulta:")
print(query_avg_prog)
print("\nResultado:")
df_avg_prog = pd.read_sql(query_avg_prog, conn)
print(df_avg_prog.to_string())
print("\n💡 Nota: AVG excluye valores NULL")

## SLIDE 9-10: COUNT()

**COUNT()** devuelve el número de filas.

**Variantes:**
- COUNT(*): Cuenta TODAS las filas (incluyendo NULLs)
- COUNT(columna): Cuenta solo filas donde columna NO es NULL

In [ ]:
print("\n" + "="*70)
print("SLIDE 9-10: COUNT()")
print("="*70 + "\n")

# COUNT(*) vs COUNT(columna)
query_count = '''
SELECT 
    COUNT(*) as total_registros,
    COUNT(programa) as registros_con_programa,
    COUNT(DISTINCT programa) as programas_unicos
FROM evaluaciones
'''

print("Consulta: Conteos diferentes")
print(query_count)
print("\nResultado:")
df_count = pd.read_sql(query_count, conn)
print(df_count.to_string())

In [ ]:
# COUNT por modalidad
query_count_modal = '''
SELECT 
    modalidad,
    COUNT(*) as total_estudiantes
FROM evaluaciones
GROUP BY modalidad
ORDER BY total_estudiantes DESC
'''

print("\n" + "-"*70)
print("Total de estudiantes por modalidad:")
print("-"*70)
print("\nConsulta:")
print(query_count_modal)
print("\nResultado:")
df_count_modal = pd.read_sql(query_count_modal, conn)
print(df_count_modal.to_string())

## SLIDE 11-15: Actividad Guiada - Reportes de Desempeño por Programa

In [ ]:
print("\n" + "="*70)
print("ACTIVIDAD GUIADA: Reportes de Desempeño por Programa")
print("="*70 + "\n")

print("📋 Situación Problema:")
print("Eres analista de datos de un servicio público de empleo.")
print("Te solicitan preparar un resumen con indicadores de programas.\n")

# Consulta 1: Promedio por programa
print("✓ CONSULTA 1: Promedio de calificaciones por programa")
print("-"*70)

q1 = '''
SELECT 
    programa,
    ROUND(AVG(calificacion), 2) as promedio,
    COUNT(*) as total_evaluaciones
FROM evaluaciones
GROUP BY programa
ORDER BY promedio DESC
'''

print("SQL:")
print(q1)
print("\nResultado:")
df_q1 = pd.read_sql(q1, conn)
print(df_q1.to_string())

In [ ]:
# Consulta 2: Total por modalidad
print("\n✓ CONSULTA 2: Total de estudiantes por modalidad")
print("-"*70)

q2 = '''
SELECT 
    modalidad,
    COUNT(*) as total_estudiantes,
    COUNT(DISTINCT programa) as programas_ofrecidos
FROM evaluaciones
GROUP BY modalidad
'''

print("SQL:")
print(q2)
print("\nResultado:")
df_q2 = pd.read_sql(q2, conn)
print(df_q2.to_string())

In [ ]:
# Consulta 3: Min y Max por programa
print("\n✓ CONSULTA 3: Nota mínima y máxima por programa")
print("-"*70)

q3 = '''
SELECT 
    programa,
    MIN(calificacion) as minima,
    MAX(calificacion) as maxima,
    ROUND(AVG(calificacion), 2) as promedio,
    MAX(calificacion) - MIN(calificacion) as rango
FROM evaluaciones
GROUP BY programa
ORDER BY rango DESC
'''

print("SQL:")
print(q3)
print("\nResultado:")
df_q3 = pd.read_sql(q3, conn)
print(df_q3.to_string())
print("\n💡 'Rango' muestra la dispersión de calificaciones")

In [ ]:
# Consulta 4: Programas con más de 2 estudiantes
print("\n✓ CONSULTA 4: Programas con más de 2 estudiantes")
print("-"*70)

q4 = '''
SELECT 
    programa,
    COUNT(*) as total_estudiantes,
    ROUND(AVG(calificacion), 2) as promedio
FROM evaluaciones
GROUP BY programa
HAVING COUNT(*) > 2
ORDER BY total_estudiantes DESC
'''

print("SQL (nota: HAVING en lugar de WHERE):")
print(q4)
print("\nResultado:")
df_q4 = pd.read_sql(q4, conn)
print(df_q4.to_string())
print("\n💡 HAVING filtra DESPUÉS de agrupar (veremos en sesión siguiente)")

## SLIDE 17-20: Actividad Práctica Autónoma

In [ ]:
print("\n" + "="*70)
print("ACTIVIDAD PRÁCTICA AUTÓNOMA: Análisis Agrupado de Rendimiento")
print("="*70 + "\n")

print("📋 Contexto: Aplicar GROUP BY y funciones de agregación")
print("sobre datos educativos para generar reportes segmentados.\n")

# Ejercicio 1: Total por modalidad
print("Ejercicio 1: Total de estudiantes por modalidad")
print("-"*70)

ej1 = '''
SELECT 
    modalidad,
    COUNT(*) as total_estudiantes
FROM evaluaciones
GROUP BY modalidad
'''

df_ej1 = pd.read_sql(ej1, conn)
print("\nResultado:")
print(df_ej1.to_string())

In [ ]:
# Ejercicio 2: Min/Max por programa
print("\nEjercicio 2: Calificación mínima y máxima por programa")
print("-"*70)

ej2 = '''
SELECT 
    programa,
    MIN(calificacion) as minima,
    MAX(calificacion) as maxima,
    MAX(calificacion) - MIN(calificacion) as dispersión
FROM evaluaciones
GROUP BY programa
ORDER BY dispersión DESC
'''

df_ej2 = pd.read_sql(ej2, conn)
print("\nResultado:")
print(df_ej2.to_string())
print("\n¿Qué programa muestra mayor dispersión entre min y máx?")
print(f"Respuesta: {df_ej2.iloc[0]['programa']} (dispersión: {df_ej2.iloc[0]['dispersión']})")

In [ ]:
# Ejercicio 3: Programas con más de 3 estudiantes
print("\nEjercicio 3: Programas con más de 3 estudiantes")
print("-"*70)

ej3 = '''
SELECT 
    programa,
    COUNT(*) as total,
    ROUND(AVG(calificacion), 2) as promedio
FROM evaluaciones
GROUP BY programa
HAVING COUNT(*) > 3
ORDER BY total DESC
'''

df_ej3 = pd.read_sql(ej3, conn)
print("\nResultado:")
if len(df_ej3) > 0:
    print(df_ej3.to_string())
else:
    print("No hay programas con más de 3 estudiantes")
    
print("\n(Cambiando a > 2 para ver datos):")
df_ej3b = pd.read_sql(ej3.replace('> 3', '> 2'), conn)
print(df_ej3b.to_string())

## SLIDE 21-22: Análisis e Interpretación

In [ ]:
print("\n" + "="*70)
print("SLIDE 21-22: Reflexión sobre Resultados")
print("="*70 + "\n")

# Crear reporte completo
query_reporte = '''
SELECT 
    programa,
    modalidad,
    COUNT(*) as estudiantes,
    ROUND(AVG(calificacion), 2) as promedio,
    MIN(calificacion) as minimo,
    MAX(calificacion) as maximo
FROM evaluaciones
GROUP BY programa, modalidad
ORDER BY programa, modalidad
'''

print("Reporte Cruzado: Programa x Modalidad")
print("-"*70)
print("\nConsulta con GROUP BY en dos columnas:")
print(query_reporte)
print("\nResultado:")
df_reporte = pd.read_sql(query_reporte, conn)
print(df_reporte.to_string())

In [ ]:
# Análisis de preguntas
print("\n" + "-"*70)
print("Preguntas de Análisis:")
print("-"*70)

# Pregunta 1
print("\n1. ¿Qué programa tiene mejor promedio?")
mejor_prog = df_reporte.loc[df_reporte['promedio'].idxmax()]
print(f"   {mejor_prog['programa']}: {mejor_prog['promedio']}")

# Pregunta 2
print("\n2. ¿Qué modalidad tiene más estudiantes?")
query_modal = pd.read_sql(
    "SELECT modalidad, COUNT(*) as total FROM evaluaciones GROUP BY modalidad ORDER BY total DESC",
    conn
)
print(f"   {query_modal.iloc[0]['modalidad']}: {int(query_modal.iloc[0]['total'])} estudiantes")

# Pregunta 3
print("\n3. ¿Cuál es el rango de calificaciones global?")
minima = df['calificacion'].min()
maxima = df['calificacion'].max()
print(f"   Mínimo: {minima}, Máximo: {maxima}, Rango: {maxima - minima}")

## SLIDE 24: Preguntas de Cierre

In [ ]:
print("\n" + "="*70)
print("PREGUNTAS DE CIERRE")
print("="*70 + "\n")

preguntas = [
    "1. ¿Cuál es la diferencia entre MIN() y MAX()?\n   MIN retorna el valor mínimo, MAX el máximo de una columna.",
    
    "2. ¿Qué precauciones al interpretar AVG()?\n   AVG excluye NULLs, considera tamaño del grupo, puede no ser representativo.",
    
    "3. ¿Por qué es importante GROUP BY correcto?\n   Sin GROUP BY completo, algunos motores devuelven errores; otros resultados impredecibles.",
    
    "4. ¿Cuándo usar COUNT(*) vs COUNT(columna)?\n   COUNT(*): todas filas, COUNT(col): solo donde col no es NULL.",
    
    "5. ¿Cómo validar resultados agrupados?\n   Verificar magnitud, comparar con conteos manuales, validar NULLs."
]

for pregunta in preguntas:
    print(pregunta)
    print()

## Resumen - Conceptos Clave

In [ ]:
print("\n" + "="*70)
print("RESUMEN: FUNCIONES DE AGREGACIÓN Y GROUP BY")
print("="*70 + "\n")

resumen = """
✓ MIN() / MAX():
  - Retornan valor mínimo/máximo de una columna
  - Útil para extremos, rangos, dispersión
  - Funcionan con números, fechas, texto

✓ AVG():
  - Calcula promedio aritmético
  - Excluye NULLs automáticamente
  - Considera todos los registros del grupo

✓ COUNT():
  - COUNT(*): todas las filas
  - COUNT(columna): filas donde columna no es NULL
  - COUNT(DISTINCT): valores únicos

✓ GROUP BY:
  - Agrupa registros por categoría(s)
  - Debe incluir todas columnas no agregadas
  - Actúa ANTES de HAVING (filtrado post-agregación)

✓ BUENAS PRÁCTICAS:
  - Usar ROUND() para decimales consistentes
  - Validar conteos y extremos
  - Considerar el tamaño del grupo
  - Documentar qué significa cada métrica
"""

print(resumen)
print("="*70)

## Cierre

In [ ]:
conn.close()
print("\n✓ Conexión cerrada")
print("\n¡Fin de la Sesión 4!")
print("\n📚 Próxima sesión: GROUP BY + HAVING (Filtrado de resultados agrupados)")